<a href="https://colab.research.google.com/github/aiza-snflwer/cardiometabolic-risk-prediction/blob/main/T2D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

data = pd.read_csv(file_name)   # IMPORTANT: T2D is CSV, not excel

print(data.shape)
data.head()



(768, 6)


In [ ]:
# Features (inputs)
X = data[['Glucose', 'BloodPressure', 'BMI', 'Age']]  # these are columns from your CSV

# Target (output)
y = data['Outcome']  # 0 = no diabetes, 1 = diabetes

from sklearn.model_selection import train_test_split

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.impute import SimpleImputer
import numpy as np

# Initialize the imputer to fill missing values with the mean
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')

# Fit the imputer on the training data and transform both training and testing data
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

from sklearn.linear_model import LogisticRegression

# Create the model
model = LogisticRegression(max_iter=1000) # Increased max_iter for convergence

# Train the model with imputed data
model.fit(X_train_imputed, y_train)

# Predict on the testing set using the imputed test data
y_pred = model.predict(X_test_imputed)

# Check accuracy
from sklearn.metrics import accuracy_score
print("Accuracy:", accuracy_score(y_test, y_pred))

# Predict probabilities using the imputed test data
y_prob = model.predict_proba(X_test_imputed)  # returns two columns: [prob_no, prob_yes]

# Get probability of diabetes (1)
risk_percentage = y_prob[:, 1] * 100  # convert to percentage

# Look at the first 5 patients
print(risk_percentage[:5])

Accuracy: 0.7532467532467533
[29.01926706 19.07916675 12.32012998  9.36680859 47.86715252]


In [ ]:
import json

all_patients_json = []

for i in range(len(X_test)):
    patient_metrics = X_test.iloc[i]
    risk = risk_percentage[i]

    # Determine risk level
    if risk > 50:
        level = "High"
    elif risk > 20:
        level = "Moderate"
    else:
        level = "Low"

    # Contributing factors
    factors = []
    if patient_metrics['Glucose'] > 125:
        factors.append({"factor": "High Glucose", "impact": "High", "explanation": "Elevated blood sugar increases diabetes risk."})
    if patient_metrics['BMI'] > 25:
        factors.append({"factor": "High BMI", "impact": "Moderate", "explanation": "Overweight contributes to insulin resistance."})
    if patient_metrics['Age'] > 40:
        factors.append({"factor": "Age", "impact": "Moderate", "explanation": "Risk increases with age."})
    if patient_metrics['BloodPressure'] > 80:
        factors.append({"factor": "High Blood Pressure", "impact": "Moderate", "explanation": "Hypertension increases cardiovascular risk."})

    patient_json = {
        "metrics": {
            "Glucose": patient_metrics['Glucose'],
            "BloodPressure": patient_metrics['BloodPressure'],
            "BMI": patient_metrics['BMI'],
            "Age": patient_metrics['Age']
        },
        "diseases": [
            {
                "name": "Type 2 Diabetes",
                "risk_percentage": round(risk, 1),
                "risk_level": level
            }
        ],
        "contributing_factors": factors
    }

    all_patients_json.append(patient_json)

# Save to JSON file
with open("t2d_predictions_with_factors.json", "w") as f:
    json.dump(all_patients_json, f, indent=2)

print("JSON with contributing factors created successfully!")

JSON with contributing factors created successfully!
